## 1. Setup
Start in the repo root so imports and relative paths work in VS Code and Jupyter. The notebook normally starts in `pipeline/notebooks/`.

In [ ]:
import os
from pathlib import Path

if not Path("pipeline/event_model/settings.json").is_file():
    os.chdir("../..")

from pipeline.event_model import notebook as em

%matplotlib inline

## 2. Settings
This starts from `pipeline/event_model/settings.json` and points it at the team's voiceovers. Each raw voiceover and its noisy copy count as one recording, and the events get mixed into the raw one. Put `VO1.wav`, `VO2.wav`, `VO3.wav` and their `_noise` copies in `data/test-data-prelim/` first (it's gitignored).

In [ ]:
settings = em.voiceover_settings("data/test-data-prelim")
run_dir = Path("results/2026-10-31-le-event-model-voiceovers")

## 3. Train or load
Train once, then load the saved model on later runs. These are tuned thresholds and tune scores; held-out test scores come in the comparison.

In [ ]:
model, settings = em.train_or_load(settings, run_dir)

## 4. One test clip
Rebuild the first clip from the saved split's held-out recording and detect its inserted events. The spectrogram shows 20 Hz to 20 kHz; green marks hits, orange marks missed labels, and red marks false alarms using the model's scoring rules.

In [ ]:
audio, record, detected = em.make_test_clip(model, settings, run_dir)
clip_figure = em.plot_test_clip(audio, record, detected, settings)

## 5. Four-version comparison
This scores the same held-out cases as clean, filtered, known filter undone, and AI estimate undone, using the Python inverse. Without the filter ID model below, it skips the AI estimate version. A score shows as `n/a` when there's nothing to divide by.

In [ ]:
filter_id_path = Path("results/2026-10-26-le-filter-id-hp-lp-bandpass/model.joblib")
comparison = em.compare_versions(run_dir, filter_id_path)
f1_figure = em.show_comparison(comparison)

## 6. Your own WAV
Replace the example path with any mono 192 kHz WAV to list its detected events. This real file has no event labels, so the list is detections only.

In [ ]:
events = em.detect_file(run_dir, "data/test-data-prelim/VO1.wav")
em.print_events(events)